# 02 Custom Video Inference

Runnable notebook for lightweight qualitative validation on user-provided videos.

- Uses the same bootstrap path as D1: `bash setup_colab.sh --with-models`
- Keeps all project logic in repository scripts and modules
- Saves each run into a dedicated subdirectory under `results/qualitative/`


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
REPO_URL = 'https://github.com/ChiCoTheLaAnh/ProjectFinalCS415.git'
REPO_DIR = '/content/ProjectFinalCS415'

!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"


In [ ]:
%cd /content/ProjectFinalCS415
!bash setup_colab.sh --with-models


In [ ]:
import shutil
import subprocess
import sys
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('Torch CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())

nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi is not None:
    gpu_name = subprocess.run(
        [nvidia_smi, '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    print('GPU:', gpu_name)

if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime is required for custom video validation.')

import groundingdino
import sam2

print('GroundingDINO import OK:', groundingdino.__file__)
print('SAM2 import OK:', sam2.__file__)


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/cv-final-project')
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
INPUT_DIR = DRIVE_ROOT / 'inputs'
OUTPUT_ROOT = DRIVE_ROOT / 'results' / 'qualitative' / 'custom_video_runs'
GROUNDING_CKPT = str(CHECKPOINT_DIR / 'groundingdino_swint_ogc.pth')
SAM2_CKPT = str(CHECKPOINT_DIR / 'sam2.1_hiera_small.pt')

INPUT_VIDEO = '/content/drive/MyDrive/cv-final-project/inputs/custom_video_01.mp4'
VIDEO_PROMPT = 'person'
RUN_NAME = 'custom_run_01'
MAX_FRAMES = 60
DEVICE = 'cuda'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Input video:', INPUT_VIDEO)
print('Prompt:', VIDEO_PROMPT)
print('Run name:', RUN_NAME)
print('Output root:', OUTPUT_ROOT)


In [ ]:
for required_path in (Path(GROUNDING_CKPT), Path(SAM2_CKPT), Path(INPUT_VIDEO)):
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required file: {required_path}')
    print(required_path, required_path.stat().st_size, 'bytes')


In [ ]:
%cd /content/ProjectFinalCS415
import subprocess

cmd = [
    'python3', 'scripts/run_custom_video.py',
    '--config', 'configs/base.yaml',
    '--input_video', INPUT_VIDEO,
    '--prompt', VIDEO_PROMPT,
    '--output_dir', str(OUTPUT_ROOT),
    '--run_name', RUN_NAME,
    '--grounding_ckpt', GROUNDING_CKPT,
    '--sam2_ckpt', SAM2_CKPT,
    '--device', DEVICE,
]
if MAX_FRAMES is not None:
    cmd.extend(['--max_frames', str(MAX_FRAMES)])
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import json
import sys
from IPython.display import Video, display

sys.path.insert(0, '/content/ProjectFinalCS415')
from src.eval.failure_analysis import summarize_failures

RUN_DIR = OUTPUT_ROOT / RUN_NAME
video_overlay = RUN_DIR / 'smoke_video_overlay.mp4'
video_summary_path = RUN_DIR / 'run_summary.json'

if not video_overlay.exists():
    raise FileNotFoundError(f'Missing video overlay: {video_overlay}')
if not video_summary_path.exists():
    raise FileNotFoundError(f'Missing video summary: {video_summary_path}')

display(Video(str(video_overlay), embed=True))
with video_summary_path.open('r', encoding='utf-8') as handle:
    video_summary = json.load(handle)
for required_field in ('model_stack', 'runtime_sec', 'artifacts'):
    if required_field not in video_summary:
        raise KeyError(f'Missing required video summary field: {required_field}')
triage = summarize_failures(video_summary)
print(json.dumps(video_summary, indent=2))
print('video_mode:', video_summary['artifacts'].get('video_mode'))
if 'fallback_reason' in video_summary['artifacts']:
    print('fallback_reason:', video_summary['artifacts']['fallback_reason'])
print('failure_triage:', json.dumps(triage, indent=2))
